In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [18]:
# Phase 1: Data Understanding
df=pd.read_csv(r"F:\Desktop\python_dataset\AB_NYC_2019.csv")
df.shape
df.head()
df.tail()
df.info
# # Display Random Sample Records
df.sample(5)
# # Quick Summary Statistics
df.describe()
# df.describe(include="object")#For categorical summary

,id,host_id,latitude,longitude,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
count,4.889500e+04,4.889500e+04,48895.000000,48895.000000,48895.000000,48895.000000,48895.000000,38843.000000,48895.000000,48895.000000
mean,1.901714e+07,6.762001e+07,40.728949,-73.952170,152.720687,7.029962,23.274466,1.373221,7.143982,112.781327
std,1.098311e+07,7.861097e+07,0.054530,0.046157,240.154170,20.510550,44.550582,1.680442,32.952519,131.622289
min,2.539000e+03,2.438000e+03,40.499790,-74.244420,0.000000,1.000000,0.000000,0.010000,1.000000,0.000000
25%,9.471945e+06,7.822033e+06,40.690100,-73.983070,69.000000,1.000000,1.000000,0.190000,1.000000,0.000000
50%,1.967728e+07,3.079382e+07,40.723070,-73.955680,106.000000,3.000000,5.000000,0.720000,1.000000,45.000000
75%,2.915218e+07,1.074344e+08,40.763115,-73.936275,175.000000,5.000000,24.000000,2.020000,2.000000,227.000000
max,3.648724e+07,2.743213e+08,40.913060,-73.712990,10000.000000,1250.000000,629.000000,58.500000,327.000000,365.000000


In [ ]:
# Phase 2: Data Quality Assessment
# Missing Values
df.isnull().sum().sort_values(ascending=False)
# # # Duplicate Records
df.duplicated().sum()#--no duplicate
# Detect unrealistiv Values(Price, Nights)
# Zeroe Price Listings
(df["price"]==0).sum()
# Extremely High Prices
df["price"].describe()
# Look at high outliers:
df[df["price"]>1000].shape

#Availability Issues (Extreme or Unusual Patterns)
df["availability_365"].describe()
(df["availability_365"]==0).sum()


np.int64(11)

2.5 Summary of Data Quality Issues
Common issues in this dataset:
Missing values in last_review and reviews_per_month
Some listings have price = 0 (unrealistic)
Extreme pricing outliers (very high values)
Potential inconsistency: listings with reviews but missing last_review

In [ ]:
# Phase 3 :Data Cleaning
# 3.1 Remove Duplicate Records
df=df.drop_duplicates()
# 3.2 Handle Missing Values
df["reviews_per_month"]=df["reviews_per_month"].fillna(0)
df["last_review"]=pd.to_datetime(df["last_review"],errors="coerce")
# optional feature:
df["has_review"]=np.where(df["last_review"].isnull(),0,1)

# 3.3 Remove Unrealistic Price Values
# Remove price = 0
df=df[df["price"]>0]

# 3.4 Treat Outliers in Price
upper_limit=df["price"].quantile(0.99)
df=df[df["price"]<=upper_limit]
print("Upper price linit used:",upper_limit)

# 3.5 Validate Cleaning
df.isnull().sum()
df.shape

In [39]:
# Phase 4: Feature Preparation
# 4.1 Convert Categorical Variables (Where needed)
df["room_type"]=df["room_type"].astype("category")
df["neighbourhood_group"]=df["neighbourhood_group"].astype("category")
df["neighbourhood"]=df["neighbourhood"].astype("category")

# 4.2 Derived Features (Very Useful for EDA)
df["price_log"]=np.log1p(df["price"])
df["availability_category"]=pd.cut(
    df["availability_365"],
    bins=[-1,0,60,180,365],
    labels=["0 (Not Available)","1-60(Low)","61-180(Medium)","181-365(High)"]
)
df["review_activity"]=pd.cut(
    df["number_of_reviews"],
    bins=[-1,0,10,50,200,df["number_of_reviews"].max()],
    labels=["0","1-10","11-50","51-200","200+"]
)
df["min_nights_category"]=pd.cut(
    df["minimum_nights"],
    bins=[0,1,3,7,30,df["minimum_nights"].max()],
    labels=["1","2-3","4-7","8-30","30+"]
)
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,...,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,has_review,price_log,availability_category,review_activity,min_nights_category
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,...,9,2018-10-19,0.21,6,365,1,5.010635,181-365(High),1-10,1
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,...,45,2019-05-21,0.38,2,355,1,5.420535,181-365(High),11-50,1
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,...,0,NaT,0.00,1,365,0,5.017280,181-365(High),0,2-3
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,...,270,2019-07-05,4.64,1,194,1,4.499810,181-365(High),200+,1
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,...,9,2018-11-19,0.10,1,0,1,4.394449,0 (Not Available),1-10,8-30
